# FINS exp51 — weak-form FINS on KdV under noise (Colab, GPU)

The 60 KdV runs of the registered noise study in weak form (`exp51_noise_weak.py --pdes kdv --parts weak`):
6 noise levels (0, 1e-3, 1e-2, 3e-2, 5e-2, 1e-1) x seeds 42-51, exact KdV solver, no training. Everything else of
exp51 runs in the sandbox. On one CPU thread a run takes ~8 min; here the solver runs on the GPU.

Runtime: T4 GPU. Upload the new `fins_repo_full.zip` from the chat to My Drive first (the setup cell takes the newest
`fins_repo_full*.zip` on Drive that contains `experiments/exp51_noise_weak.py`). Outputs go to
`MyDrive/fins_runs/results/noise_weak_runs`; after a disconnect rerun cells 1 and 2: finished runs are skipped.

Cells: 1 setup; 2 the run; 3 progress; 4 zip of the outputs (npz and logs) for download.

In [ ]:
# 1. setup: Drive, repository from the newest fins_repo_full*.zip on Drive that contains exp51, dependencies
import os, sys, subprocess, zipfile, glob, time, shutil
from google.colab import drive
drive.mount('/content/drive')
os.environ['FINS_RESULTS_DIR'] = '/content/drive/MyDrive/fins_runs/results'
os.makedirs(os.environ['FINS_RESULTS_DIR'], exist_ok=True)
cands = sorted(glob.glob('/content/drive/MyDrive/fins_repo_full*.zip') + glob.glob('/content/drive/MyDrive/*/fins_repo_full*.zip'),
               key=os.path.getmtime, reverse=True)
ZIP_PATH = None
for z in cands:
    has = 'fins_repo/experiments/exp51_noise_weak.py' in zipfile.ZipFile(z).namelist()
    print(f'{z}  {time.ctime(os.path.getmtime(z))}  {os.path.getsize(z)} bytes  exp51: {has}')
    if has and ZIP_PATH is None:
        ZIP_PATH = z
if ZIP_PATH is None:
    raise SystemExit('no fins_repo_full*.zip on Drive contains experiments/exp51_noise_weak.py: upload the zip from '
                     'the chat to My Drive and rerun this cell')
print('using', ZIP_PATH)
shutil.rmtree('/content/fins', ignore_errors=True)             # never reuse a tree unpacked from an older zip
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/fins')
REPO = '/content/fins/fins_repo'
EXPDIR = os.path.join(REPO, 'experiments')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neuraloperator'], check=False)
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if not torch.cuda.is_available():
    print('NO GPU: Runtime -> Change runtime type -> T4 GPU, then rerun this cell')

In [ ]:
# 2. run: 60 weak-form KdV runs (6 noise levels x seeds 42-51); one line per finished run
RUNS = os.path.join(os.environ['FINS_RESULTS_DIR'], 'noise_weak_runs')
cmd = [sys.executable, '-u', 'exp51_noise_weak.py', 'run', '--pdes', 'kdv', '--parts', 'weak', '--runs', RUNS]
t0 = time.time()
with subprocess.Popen(cmd, cwd=EXPDIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
    for line in p.stdout:
        print(line, end='', flush=True)
print(f'exit {p.returncode} after {time.time() - t0:.0f} s')

In [ ]:
# 3. progress (safe while cell 2 runs, or after it stopped): finished runs per condition and the last line of each log
RUNS = os.path.join(os.environ['FINS_RESULTS_DIR'], 'noise_weak_runs')
for d in sorted(glob.glob(os.path.join(RUNS, '*'))):
    done = sorted(os.path.basename(f) for f in glob.glob(os.path.join(d, 'weak_kdv_*.npz')))
    print(os.path.basename(d), len(done), 'of 10:', ' '.join(f.split('_')[-1][:-4] for f in done))
    for log in sorted(glob.glob(os.path.join(d, 'logs', 'weak_kdv_*.log'))):
        lines = [l for l in open(log).read().splitlines() if l.strip()]
        print('   ', os.path.basename(log), '|', lines[-1][:150] if lines else '')

In [ ]:
# 4. zip the outputs (npz + logs) and download; a copy stays on Drive
from google.colab import files
RUNS = os.path.join(os.environ['FINS_RESULTS_DIR'], 'noise_weak_runs')
arc = shutil.make_archive('/content/fins_exp51_kdv_weak', 'zip', RUNS)
shutil.copy(arc, '/content/drive/MyDrive/fins_exp51_kdv_weak.zip')
print(arc, os.path.getsize(arc), 'bytes; a copy is on Drive')
files.download(arc)